# Einleitung

Dieses Projekt analysiert globale Waldbrandereignisse anhand satellitenbasierter Daten des NASA FIRMS Systems.
Der Fokus liegt auf der Untersuchung räumlicher Muster und der Intensität von Brandereignissen innerhalb eines kurzen Zeitraums (letzte 24 Stunden).

Der Datensatz enthält georeferenzierte Punktdaten, wobei jede Zeile ein detektiertes Feuerereignis repräsentiert.
Wichtige Variablen wie die geografische Lage (Latitude/Longitude), die Detektionssicherheit (confidence) sowie die Feuerintensität (Fire Radiative Power, FRP) werden genutzt, um die Verteilung und Eigenschaften der Brände zu analysieren.

Das Ziel dieses Projekts ist es, eine automatisierte Pipeline zu entwickeln, die Waldbranddaten verarbeitet, relevante Informationen herausfiltert, eine gezielte Analyse ermöglicht und die Ergebnisse übersichtlich auf einer interaktiven Karte visualisiert.

Die Forschungsfrage, die mit diesem Projekt beantwortet werden soll, ist: *Wo befinden sich die intensivsten und zuverlässig detektierten Waldbrände weltweit innerhalb der letzten 24 Stunden?*

Als Leitfragen zur Beantwortung der Forschungsfrage dienen:
1. **Intensität**: Wo treten die stärksten Waldbrände (höchste Fire Radiative Power) auf?
2. **Datenqualität**: Wie verteilt sich die Detektionssicherheit (confidence) über die erfassten Waldbrände?
3. **Räumliche Analyse**: Sind Waldbrandereignisse in bestimmten Regionen der Welt konzentriert?
4. **Zeitliche Analyse**: Wie viele Waldbrandereignisse wurden im betrachteten Zeitraum detektiert?

### Schritt 1: Importe, Paths & Überblick verschaffen

In [11]:
#Importe
from pathlib import Path
import pandas as pd
import geopandas as gpd

#Pfade
data_dir = Path("../data") #heisst "gehe eine Orderstruktur höher" und dann in "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
csv_path = raw_dir / "SUOMI_VIIRS_C2_Global_24h.csv"

#Rohdaten laden
data = pd.read_csv(csv_path)

#Inhalte anzeigen & Überblick verschaffen
print(data.head())
print() #Zeilenumbruch
print(data.info())
print() #Zeilenumbruch


   latitude  longitude  bright_ti4  scan  track    acq_date  acq_time  \
0 -18.35942   26.65817      302.52  0.44   0.46  2026-05-18         1   
1 -18.35993   26.66234      309.22  0.44   0.46  2026-05-18         1   
2 -18.36387   26.45063      332.84  0.43   0.46  2026-05-18         1   
3 -18.36698   26.47523      320.07  0.43   0.46  2026-05-18         1   
4 -18.36750   26.47932      321.52  0.43   0.46  2026-05-18         1   

  satellite confidence version  bright_ti5   frp daynight  
0         N    nominal  2.0NRT      289.85  1.19        N  
1         N    nominal  2.0NRT      289.42  1.53        N  
2         N    nominal  2.0NRT      292.60  4.09        N  
3         N    nominal  2.0NRT      292.31  2.85        N  
4         N    nominal  2.0NRT      291.41  2.85        N  

<class 'pandas.DataFrame'>
RangeIndex: 39787 entries, 0 to 39786
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    397

**Zusammenfassung Schritt 1:** _data.head()_ hat uns die ersten 5 Reihen ausgegeben. Daraus können wir ableiten, dass für die Forschungsfrage sind lediglich die Spalten "latitude", "longitude", "acq_date", "confidence" und "frp" relevant sind. Daher selektieren wir diese in Schritt 2. Die Header sollten ausserdem noch etwas verständlicher unbenannt werden. Dafür stütze ich mich auf die Informationen im User Guide, der unter den Rohdaten abgelegt ist. Zudem hat und _data.info()_ gezeigt, dass acq_date momentan noch ein String ist. Das müssen wir ebenfalls ändern.

### Step 2: Bereinigung der Rohdaten

In [13]:
relevant_data = data[["latitude", "longitude", "acq_date", "confidence", "frp"]].copy() #relevante Spalten auswählen und eine echte Kopie herstellen vom original dataframe
print() #Zeilenumbruch
print(relevant_data.head()) #reduzierter Dataframe anzeigen
print() #Zeilenumbruch
print(relevant_data.isna().sum()) #überprüfen, ob fehlende Werte (NaN) bestehen
print() #Zeilenumbruch

#Header unbenennen:
relevant_data = relevant_data.rename(columns={
    "latitude": "lat",
    "longitude": "lon",
    "acq_date": "detection_date",
    "confidence": "detection_confidence",
    "frp": "fire_radiative_power"
})

print(relevant_data.head()) #neue Header anzeigen lassen
print() #Zeilenumbruch

relevant_data["detection_date"] = pd.to_datetime(relevant_data["detection_date"]) #detection_date in Datum umwandeln
print(relevant_data.info()) #schauen ob der string erfolgreich umgewandelt wurde
print() #Zeilenumbruch




   latitude  longitude    acq_date confidence   frp
0 -18.35942   26.65817  2026-05-18    nominal  1.19
1 -18.35993   26.66234  2026-05-18    nominal  1.53
2 -18.36387   26.45063  2026-05-18    nominal  4.09
3 -18.36698   26.47523  2026-05-18    nominal  2.85
4 -18.36750   26.47932  2026-05-18    nominal  2.85

latitude      0
longitude     0
acq_date      0
confidence    0
frp           0
dtype: int64

        lat       lon detection_date detection_confidence  \
0 -18.35942  26.65817     2026-05-18              nominal   
1 -18.35993  26.66234     2026-05-18              nominal   
2 -18.36387  26.45063     2026-05-18              nominal   
3 -18.36698  26.47523     2026-05-18              nominal   
4 -18.36750  26.47932     2026-05-18              nominal   

   fire_radiative_power  
0                  1.19  
1                  1.53  
2                  4.09  
3                  2.85  
4                  2.85  

<class 'pandas.DataFrame'>
RangeIndex: 39787 entries, 0 to 39786
Dat

**Zusammenfassung Schritt 2:**
Wir machen von unserer Selektion des ursprünglichen Dataframes eine Kopie, um Komplikationen bei späteren Modifikationen an unserem Subset zu verhindern. *relevant_data.isna().sum()* hat ergeben, dass es bei unserem Datensatz keine fehlenden Werte gibt. Daher müssen wir an dieser Stelle nichts bereinigen. Das detection_date, das bisher noch als string erfasst war wurde jetzt in ein Datum umegwandelt.

Gemäss User Guide sind die Header wie folgt zu interpretieren:
* detection_date = YYYYMMDD Erfassungsdatum in year (YYYY), month (MM) and day (DD)
* detection_confidence =  Branderkennungs-Konfidenz (“L”=low, “N”=nominal, “H”=high)
* fire_radiative_power Strahlungsleistung des Feuers in megawatt (Mass für thermische Intensität)

## Schritt 3: Datenanalyse

##### 3.1. Statistische Übersicht

Zusammenfassung Schritt 3.1.

##### 3.2. Statistische Übersicht

##### 3.3 Zeitliche Analyse

##### 3.4. Räumliche Analyse

### Schritt 4: Datenvisualisiserung

##### 4.1. **Intensität**: Wo treten die stärksten Waldbrände (höchste Fire Radiative Power) auf?


##### 4.2. **Datenqualität**: Wie verteilt sich die Detektionssicherheit (confidence) über die erfassten Waldbrände?


##### 4.3. **Räumliche Verteilung**: Sind Waldbrandereignisse in bestimmten Regionen der Welt konzentriert?

##### 4.4. **Zeitliche Verteilung**: Wie viele Waldbrandereignisse wurden im betrachteten Zeitraum detektiert?